In [1]:
# -*- coding: utf-8 -*-
"""
气相 Cl XPAHs — 两段式 SHAP 解释脚本
=====================================
- Stage 1: 静态城市结构特征 (static_cols)                       -> 解释 bst1
- Stage 2: 动态/气象特征 (dynamic_cols) + Stage1预测(log1p)     -> 解释 bst2

前置条件: 先运行 two_stage_xpahs_pred_2019_2021_2023.py 的 main()，
在 gas.OUT_DIR 下生成: stage1_model.json / stage2_model.json / train_artifacts.json

输出: <gas.OUT_DIR>/shap_gas/
  stage1_shap_mean_abs.csv / _summary.png / _bar.png / _top15.png / _dependence_*.png
  stage2_shap_mean_abs.csv / ... (同上, 特征含 stage1_pred_log1p)
  shap_two_stage_readme.txt
"""
import os
import json
import importlib.util

import numpy as np
import pandas as pd
import xgboost as xgb
import shap

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------- 1) 导入气相两段式模块 (只定义, 不运行 main) ----------
GAS_SCRIPT = r"two_stage_xpahs_pred_2019_2021_2023.py"
spec = importlib.util.spec_from_file_location("gas_two_stage", GAS_SCRIPT)
gas = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gas)

OUT_DIR = gas.OUT_DIR
SHAP_DIR = os.path.join(OUT_DIR, "shap_gas")
os.makedirs(SHAP_DIR, exist_ok=True)

# ---------- 2) 读取训练数据, 复用气相模块预处理 ----------
print("读取训练数据:", gas.DATA_PATH)
df_raw = gas.try_read_csv(gas.DATA_PATH)
df_ok, y, groups, Xdf, artifacts = gas.preprocess_training(df_raw)

static_cols = list(artifacts.static_cols)
dynamic_cols = list(artifacts.dynamic_cols)
X_static = Xdf[static_cols].values.astype(float)
X_dyn = Xdf[dynamic_cols].values.astype(float)
print(f"static 特征数 = {len(static_cols)} | dynamic 特征数 = {len(dynamic_cols)} | 样本数 = {X_static.shape[0]}")

# ---------- 3) 加载已训练模型 ----------
art_path = os.path.join(OUT_DIR, "train_artifacts.json")
m1 = os.path.join(OUT_DIR, "stage1_model.json")
m2 = os.path.join(OUT_DIR, "stage2_model.json")
if not (os.path.exists(art_path) and os.path.exists(m1) and os.path.exists(m2)):
    raise FileNotFoundError(
        "未找到训练产物, 请先运行完整训练脚本 two_stage_xpahs_pred_2019_2021_2023.py 的 main()。\n"
        f"期望: {m1}\n      {m2}\n      {art_path}"
    )

arts = json.load(open(art_path, encoding="utf-8"))
it1, it2 = int(arts["it1"]), int(arts["it2"])

bst1 = xgb.Booster()
bst1.load_model(m1)
bst2 = xgb.Booster()
bst2.load_model(m2)
print(f"已加载模型: it1={it1}, it2={it2}")

# ---------- 4) SHAP 输出工具 ----------
def produce_shap_outputs(shap_values, X_df, tag):
    shap_df = pd.DataFrame(shap_values, columns=X_df.columns)
    mean_abs = shap_df.abs().mean().sort_values(ascending=False).reset_index()
    mean_abs.columns = ["feature", "mean_abs_shap"]
    mean_abs.to_csv(os.path.join(SHAP_DIR, f"{tag}_shap_mean_abs.csv"), index=False, encoding="utf-8-sig")

    shap.summary_plot(shap_values, X_df, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(SHAP_DIR, f"{tag}_shap_summary.png"), dpi=300, bbox_inches="tight")
    plt.close()

    shap.summary_plot(shap_values, X_df, plot_type="bar", show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(SHAP_DIR, f"{tag}_shap_bar.png"), dpi=300, bbox_inches="tight")
    plt.close()

    plot_top = mean_abs.head(15).iloc[::-1]
    plt.figure(figsize=(8, max(5, 0.35 * len(plot_top))))
    plt.barh(plot_top["feature"], plot_top["mean_abs_shap"])
    plt.xlabel("Mean |SHAP value|")
    plt.title(f"Top 15 SHAP importance ({tag})")
    plt.tight_layout()
    plt.savefig(os.path.join(SHAP_DIR, f"{tag}_shap_top15.png"), dpi=300, bbox_inches="tight")
    plt.close()

    for feat in mean_abs["feature"].head(3).tolist():
        plt.figure(figsize=(6, 4.5))
        plt.scatter(X_df[feat].values, shap_df[feat].values, alpha=0.6)
        plt.xlabel(feat)
        plt.ylabel(f"SHAP value of {feat}")
        plt.title(f"Dependence: {feat} ({tag})")
        plt.tight_layout()
        plt.savefig(os.path.join(SHAP_DIR, f"{tag}_dependence_{feat}.png"), dpi=300, bbox_inches="tight")
        plt.close()

    return mean_abs

# ---------- 5) Stage 1 SHAP (静态特征解释 bst1) ----------
print("\n[Stage 1] SHAP on static features ...")
masker1 = shap.maskers.Independent(X_static, max_samples=X_static.shape[0])
explainer1 = shap.TreeExplainer(bst1, masker1)
shap_values1 = explainer1.shap_values(X_static)
ma1 = produce_shap_outputs(shap_values1, pd.DataFrame(X_static, columns=static_cols), "stage1")

# ---------- 6) Stage 2 SHAP (动态特征 + Stage1预测 解释 bst2) ----------
print("[Stage 2] SHAP on dynamic features + stage1_pred_log1p ...")
stage1_pred_log = bst1.predict(gas.build_dmatrix(X_static), iteration_range=(0, it1)).reshape(-1, 1)
X2 = np.hstack([X_dyn, stage1_pred_log])
stage2_names = list(dynamic_cols) + ["stage1_pred_log1p"]
masker2 = shap.maskers.Independent(X2, max_samples=X2.shape[0])
explainer2 = shap.TreeExplainer(bst2, masker2)
shap_values2 = explainer2.shap_values(X2)
ma2 = produce_shap_outputs(shap_values2, pd.DataFrame(X2, columns=stage2_names), "stage2")

# ---------- 7) 汇总 ----------
readme = []
readme.append("气相两段式 SHAP 输出说明")
readme.append("=" * 40)
readme.append(f"训练数据: {gas.DATA_PATH}")
readme.append(f"样本数: {X_static.shape[0]}")
readme.append(f"Stage1 静态特征数: {len(static_cols)}")
readme.append(f"Stage2 动态特征数: {len(dynamic_cols)} (+1 stage1_pred_log1p)")
readme.append("")
readme.append("Stage1 Top10 特征 (按 mean|SHAP|):")
readme.append(ma1.head(10).to_string(index=False))
readme.append("")
readme.append("Stage2 Top10 特征 (按 mean|SHAP|):")
readme.append(ma2.head(10).to_string(index=False))
readme.append("")
readme.append("说明: Stage2 的 stage1_pred_log1p 表示第一段预测对第二段残差修正的贡献。")
with open(os.path.join(SHAP_DIR, "shap_two_stage_readme.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(readme))

print("\n完成。SHAP 结果已保存到:", SHAP_DIR)


读取训练数据: ../Data/Data_S1_Gas_training.csv
static 特征数 = 17 | dynamic 特征数 = 15 | 样本数 = 205
已加载模型: it1=716, it2=7

[Stage 1] SHAP on static features ...
[Stage 2] SHAP on dynamic features + stage1_pred_log1p ...

完成。SHAP 结果已保存到: gas_output\shap_gas
